# Bước 1 trên Kaggle: tái lập SP-Det, YOLOv8s, YOLO-World (chia 8:1:1)

Bản này giống `01_reproduce_spdet.ipynb`, chỉ thêm phần để chạy trên Kaggle. Hướng dẫn đầy đủ nằm ở `KAGGLE_SETUP.md`.

**Cài đặt notebook (bảng bên phải → Session options):**
- Accelerator: **GPU T4 x2**. Không chọn P100, vì PyTorch mới trên Kaggle không còn hỗ trợ P100.
- Internet: **On**. Cần để tải CheXagent, BERT, CLIP và trọng số YOLO.

**Input cần thêm (Add Input):**
1. Cuộc thi *VinBigData Chest X-ray Abnormalities Detection*.
2. Dataset mã nguồn `spdet-code` mà nhóm upload (thư mục có `prep.py`, `spdet_model.py`).
3. Từ phiên thứ hai trở đi: output của phiên trước (file `spdet_work.tar`).

**Chạy bằng Save Version → Save & Run All (Commit).** Mỗi phiên tối đa 12 giờ, nên notebook tự dừng trước 11 giờ.
Cuối phiên, toàn bộ tiến độ được đóng gói vào `spdet_work.tar`; phiên sau thêm file đó làm input và chạy tiếp từ chỗ dừng.

- Phiên 1: chuyển ảnh và sinh báo cáo CheXagent (phần lâu nhất).
- Phiên 2, 3: mã hoá gợi ý, train ba mô hình, chấm điểm.

Mỗi mô hình chỉ được bắt đầu khi thời gian còn lại đủ theo `EST_HOURS`. Sau phiên đầu có train, hãy sửa `EST_HOURS` theo thời gian thật trong log.

# Bước 1: Tái lập SP-Det trên VinDr-CXR (chia 8:1:1 như bài gốc)

Notebook này làm các mục **1.1, 1.2, 1.3** trong task list (Bảng 7 của review, bước 1):

| Mục | Mô hình | Đích theo Bảng 1 của bài (mAP40:95 / AP40) |
|---|---|---|
| 1.1 | SP-Det (YOLOv8s + YOLO-World + khối gợi ý) | 21.0 / 42.4 |
| 1.2 | YOLOv8s, không gợi ý | 19.4 / 40.3 |
| 1.3 | YOLO-World (yolov8s-worldv2) | 20.5 / 40.1 |

Theo review: nếu không tái lập được 21.0% thì phải ghi rõ mức lệch hoặc liên hệ tác giả.

**Bài gốc không công bố mã.** Phần SP-Det được dựng lại theo mô tả trong bài, nằm ở `spdet_model.py` và `prep.py`. Những chỗ bài không nói rõ và lựa chọn của nhóm:

1. **Gộp hộp của ba bác sĩ:** dùng WBF với IoU 0.4. Bài không nói cách gộp; đây là cờ lỗi số 3 trong review.
2. **Kích thước ảnh:** 640, vì bài không nêu.
3. **Tắt mosaic và lật trái phải cho cả ba mô hình.** Báo cáo của SP-Det là mô tả của một ảnh; mosaic ghép bốn bệnh nhân vào một ảnh, còn lật ảnh làm "tràn dịch bên trái" thành bên phải. Ba mô hình dùng chung cấu hình để so sánh công bằng.
4. **Enhancer** đặt sau SPPF (feature map cao nhất), độ sâu 2. Gamma khởi tạo 0, nên lúc bắt đầu SP-Det trùng với YOLO-World.
5. **Trích tên bệnh:** cụm danh từ (spaCy) + NegEx đơn giản + từ đồng nghĩa cho 14 lớp, thay cho "clinical word embeddings" mà bài không nêu cụ thể.
6. **Early stopping:** theo mAP của Ultralytics (patience 20), còn bài dùng val loss.

**Cần có:** 1 GPU khoảng 24 GB (bài dùng 2 × A5000; ở đây chạy 1 GPU vì khối gợi ý không chạy được với DDP của Ultralytics), dữ liệu cuộc thi Kaggle *VinBigData Chest X-ray Abnormalities Detection* (`train.csv` và thư mục `train/` chứa DICOM). Chỉ cần 4,394 ảnh có tổn thương; có thể chỉ tải các file đó.

In [ ]:
# Cài gói (chạy mỗi phiên, khoảng 2-3 phút). Không cần restart vì chưa import các gói này.
# Kaggle dùng Python 3.13, không cài được transformers 4.40 mà CheXagent cần; CheXagent chạy trong môi trường
# Python 3.11 riêng (ô mục 2). Notebook chính dùng transformers có sẵn của Kaggle cho BERT.
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("ultralytics==8.4.171", "ensemble-boxes", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "spacy",
    "transformers", "uv")
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)

## 0. Cấu hình

In [ ]:
import os, sys, json, csv, time, glob, tarfile, shutil
from pathlib import Path
import numpy as np, pandas as pd, yaml, torch

T0 = time.time()
MAX_HOURS = 11.0          # Kaggle dừng phiên ở 12 giờ và khi đó có thể mất output; chừa 1 giờ cho chấm điểm và đóng gói
def hours_left():
    return MAX_HOURS - (time.time() - T0) / 3600

def find_input(name, parent_has=None):
    """Tìm file trong /kaggle/input (sâu tối đa 4 cấp), vì đường dẫn input phụ thuộc tên dataset."""
    for d in range(5):
        for p in sorted(glob.glob("/kaggle/input/" + "*/" * d + name)):
            if parent_has is None or (Path(p).parent / parent_has).exists():
                return Path(p)
    return None

SPDET_ROOT = find_input("spdet_model.py").parent              # dataset mã nguồn spdet-code
KAGGLE_DIR = find_input("train.csv", parent_has="train").parent  # dữ liệu cuộc thi VinBigData
IMG_SOURCE = "dicom"
PNG_SRC = META_SRC = None
WORK = Path("/kaggle/temp/spdet_work")                         # đĩa tạm của phiên; cuối notebook đóng gói vào /kaggle/working
SAVE_TAR = Path("/kaggle/working/spdet_work.tar")
DEVICE = 0                 # T4 x2: chỉ dùng GPU 0, vì khối gợi ý không chạy DDP
SEEDS = [0]
SPLIT_SEED = 42
RUN = {"yolov8s": True, "yolo_world": True, "spdet": True}
EST_HOURS = {"yolov8s": 2.5, "yolo_world": 3.5, "spdet": 4.0}  # ước tính trên T4 cho 100 epoch, CHƯA ĐO; sửa theo log thật
TRAIN_BATCH = 32           # nếu báo "CUDA out of memory", đổi 16 (Ultralytics vẫn cộng dồn gradient tới 64 ảnh)
CHEX_DTYPE = "auto"        # "auto": bf16 trên GPU đời Ampere trở lên, fp16 trên T4. Nếu báo cáo in ra rỗng hoặc lặp ký tự, đổi "float32"
SCP_BOXES = True           # CheXagent-2 trả về tên bệnh kèm hộp toạ độ thay vì báo cáo văn bản. True: giữ cả toạ độ trong
                           # gợi ý (đúng pipeline của bài); False: chỉ giữ tên bệnh. Đổi được sau, không phải sinh lại báo cáo

# Khôi phục tiến độ của phiên trước (nếu đã thêm output phiên trước làm input)
prev = find_input("spdet_work.tar")
if prev and not WORK.exists():
    print("Khôi phục", prev); t = time.time()
    WORK.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(prev) as tf:
        tf.extractall(WORK.parent)
    print(f"  xong trong {time.time() - t:.0f}s")
elif not prev:
    print("Không thấy spdet_work.tar trong input: chạy từ đầu (phiên 1).")
print("SPDET_ROOT =", SPDET_ROOT, "| KAGGLE_DIR =", KAGGLE_DIR)

sys.path.insert(0, str(SPDET_ROOT)); sys.path.insert(0, str(SPDET_ROOT / "eval"))
import prep, cxr_eval

CFG = yaml.safe_load(open(SPDET_ROOT / "configs" / "common.yaml"))
TRAIN_ARGS = dict(CFG["train"])
TRAIN_ARGS.update(mosaic=0.0, fliplr=0.0, batch=TRAIN_BATCH, workers=min(8, os.cpu_count() or 4))
PNG_DIR, DS_DIR, PROMPT_DIR, RES_DIR = WORK / "png", WORK / "yolo_811", WORK / "prompts", WORK / "results_step1"
for d in (PNG_DIR, DS_DIR, PROMPT_DIR, RES_DIR):
    d.mkdir(parents=True, exist_ok=True)
import ultralytics; print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available(),
                          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert ultralytics.__version__ == CFG["ultralytics_version"], "Sai phiên bản ultralytics so với configs/common.yaml"
TRAIN_ARGS

## 1. Dữ liệu: chọn 4,394 ảnh có tổn thương, DICOM sang PNG, gộp hộp, chia 8:1:1

In [ ]:
rows = prep.read_kaggle_train(KAGGLE_DIR / "train.csv")
lesion_ids = sorted(i for i, b in rows.items() if b)
print(f"{len(rows)} ảnh trong train.csv, {len(lesion_ids)} ảnh có tổn thương (bài: 15,000 và 4,394)")
meta = (prep.convert_all(lesion_ids, KAGGLE_DIR / "train", PNG_DIR, WORK / "meta.csv", workers=os.cpu_count() or 4) if IMG_SOURCE == "dicom"
        else prep.use_existing_images(lesion_ids, PNG_SRC["train"], PNG_DIR, META_SRC["train"], WORK / "meta.csv"))
print(len(meta), "ảnh đã có PNG")

In [ ]:
fused = prep.fuse_radiologists({i: rows[i] for i in lesion_ids}, meta, iou_thr=CFG["data"]["wbf_iou_thr"])
n_raw = sum(len(rows[i]) for i in lesion_ids); n_fused = sum(len(v) for v in fused.values())
print(f"{n_raw} hộp của bác sĩ -> {n_fused} hộp sau WBF (IoU {CFG['data']['wbf_iou_thr']})")

split_file = WORK / "split_811.json"
if split_file.exists():
    splits = json.load(open(split_file))
else:
    splits = prep.split_811(lesion_ids, seed=SPLIT_SEED); json.dump(splits, open(split_file, "w"))
counts = pd.DataFrame({k: prep.class_counts(fused, v) for k, v in splits.items()}, index=prep.VINDR_CLASSES)
counts.loc["Số ảnh"] = [len(v) for v in splits.values()]
DATA_YAML = prep.write_yolo_dataset(DS_DIR, splits, fused, meta, PNG_DIR)
prep.write_gt_csv(fused, splits["test"], RES_DIR / "gt_test.csv")
counts   # số hộp mỗi lớp theo tập; lớp hiếm chỉ vài hộp ở test (cờ lỗi số 2 của review)

## 2. Gợi ý ngữ cảnh (SCP): sinh báo cáo bằng CheXagent

Bài dùng CheXagent (SigLIP-Large + Phi-2), 1 lượt hỏi với câu *"Describe the chest X-ray image"*, rồi bỏ câu dở dang cuối và đoạn lặp.
Ô dưới chạy lại được: ảnh nào đã có báo cáo trong `reports.jsonl` sẽ được bỏ qua.
Nếu tải mô hình lỗi với transformers mới, hãy chạy riêng phần này trong môi trường có `transformers==4.40.0`; phần này chỉ ghi ra file `reports.jsonl`.

In [ ]:
CHEXAGENT = "StanfordAIMI/CheXagent-2-3b"
PROMPT = "Describe the chest X-ray image"
REPORTS = WORK / "reports.jsonl"
CHEX_ENV = Path("/kaggle/temp/chex_env")      # Python 3.11 + transformers 4.40 (theo model card của CheXagent-2-3b)
CHEX_PKGS = ["torch==2.3.1", "torchvision==0.18.1", "transformers==4.40.0", "tokenizers==0.19.1", "huggingface_hub<0.26",
             "accelerate<1.0", "numpy<2", "einops", "tiktoken", "transformers_stream_generator", "pillow", "scipy",
             "matplotlib", "timm", "sentencepiece", "protobuf", "opencv-python-headless", "albumentations",
             "pyarrow", "pandas", "requests"]

# Script chạy trong môi trường riêng: nạp CheXagent, sinh báo cáo cho từng ảnh, ghi nối vào reports.jsonl
CHEX_SCRIPT = r"""
import json, sys, time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
name, prompt, png_dir, ids_file, out, dtype, deadline = sys.argv[1:8]
ids = open(ids_file).read().split()
dt = getattr(torch, dtype)
tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(name, device_map={"": 0}, trust_remote_code=True, torch_dtype=dt).to(dt).eval()
print("CheXagent da nap, dtype", dt, flush=True)
t = time.time()
for k, iid in enumerate(ids):
    if time.time() > float(deadline):
        print(f"Het gio cua phien sau {k} bao cao; phien sau se lam tiep.", flush=True)
        break
    query = tok.from_list_format([{"image": f"{png_dir}/{iid}.png"}, {"text": prompt}])
    conv = [{"from": "system", "value": "You are a helpful assistant."}, {"from": "human", "value": query}]
    x = tok.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        o = model.generate(x, attention_mask=torch.ones_like(x), pad_token_id=tok.eos_token_id, do_sample=False, num_beams=1, temperature=1.0, top_p=1.0, use_cache=True, max_new_tokens=512)[0]
    raw = tok.decode(o[x.size(1):-1])
    with open(out, "a") as f:
        f.write(json.dumps({"image_id": iid, "raw": raw}) + "\n")
    if k < 3 or k % 200 == 0:
        print(k, iid, f"| {(time.time() - t) / (k + 1):.1f}s/anh |", raw[:200], flush=True)
"""

import os, re, subprocess

def parse_grounding(raw):
    """CheXagent-2 trả về dạng '<|ref|> Lung Opacity <|/ref|> <|box|> (28,18),(42,29) <|/box|> ...'.
    Trả về [(tên, (x1, y1, x2, y2))], toạ độ theo thang 0-100 của ảnh; bỏ phần bị cắt dở ở cuối."""
    out = []
    for chunk in raw.split("<|ref|>")[1:]:
        if "<|/ref|>" not in chunk:
            continue
        name, rest = chunk.split("<|/ref|>", 1)
        for b in re.findall(r"<\|box\|>(.*?)<\|/box\|>", rest):
            v = [int(x) for x in re.findall(r"\d+", b)]
            if len(v) == 4:
                out.append((name.strip(), tuple(v)))
    return out

def to_report(raw):
    """Văn bản đưa vào BERT (SCP). Báo cáo thường: làm sạch như bài. Dạng hộp: mỗi hộp thành một câu."""
    if "<|ref|>" not in raw:
        return prep.clean_report(raw)
    sents = []
    for name, b in parse_grounding(raw):
        s = f"{name} at ({b[0]},{b[1]}),({b[2]},{b[3]})." if SCP_BOXES else f"{name}."
        if s not in sents:
            sents.append(s)
    return " ".join(sents)

def chex_dtype():
    # Bài dùng bf16. T4 không có bf16 phần cứng nên mặc định chạy fp16; ghi lại kiểu số trong bài.
    if CHEX_DTYPE != "auto":
        return CHEX_DTYPE
    return "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"

def chex_install(py, *pkgs):
    subprocess.run([sys.executable, "-m", "uv", "pip", "install", "-q", "-p", str(py), *pkgs], check=True)

# Tên module trong lỗi -> tên gói pip, cho các trường hợp khác nhau
PIP_NAME = {"cv2": "opencv-python-headless", "PIL": "pillow", "sklearn": "scikit-learn", "yaml": "pyyaml",
            "skimage": "scikit-image", "google": "protobuf"}

def missing_pkgs(log):
    """Đọc tên gói còn thiếu từ log lỗi của Python hoặc của transformers (mã remote)."""
    m = re.search(r"packages that were not found in your environment: ([^.\n]+)", log)
    if m:
        return [PIP_NAME.get(x.strip(), x.strip()) for x in m.group(1).split(",")]
    m = re.findall(r"No module named '([\w.]+)'", log)
    return [PIP_NAME.get(m[-1].split(".")[0], m[-1].split(".")[0])] if m else []

def ensure_chex_env():
    py = CHEX_ENV / "bin" / "python"
    if not py.exists():
        t = time.time(); print("Tạo môi trường Python 3.11 cho CheXagent (tải khoảng 3 GB)...")
        subprocess.run([sys.executable, "-m", "uv", "venv", "-q", "-p", "3.11", str(CHEX_ENV)], check=True)
        chex_install(py, *CHEX_PKGS)
        print(f"  xong trong {time.time() - t:.0f}s")
    return py

def run_chexagent(ids, deadline):
    # Chạy script trong môi trường riêng và in log ra notebook theo thời gian thực.
    # Mã remote của CheXagent import thêm gói không ghi trong model card: thiếu gói nào thì cài gói đó rồi chạy lại.
    py = ensure_chex_env()
    script, ids_file = WORK.parent / "chex_reports.py", WORK.parent / "chex_todo.txt"
    script.write_text(CHEX_SCRIPT); ids_file.write_text("\n".join(ids))
    # Jupyter đặt MPLBACKEND=module://matplotlib_inline...; matplotlib trong môi trường riêng không có backend đó
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}
    env.update(MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
    for attempt in range(10):
        p = subprocess.Popen([str(py), str(script), CHEXAGENT, PROMPT, str(PNG_DIR), str(ids_file), str(REPORTS), chex_dtype(),
                              str(deadline)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        tail = []
        for line in p.stdout:
            print(line, end=""); tail = (tail + [line])[-30:]
        if p.wait() == 0:
            return
        pkgs = missing_pkgs("".join(tail))
        if not pkgs:
            raise RuntimeError("CheXagent lỗi, xem log ngay phía trên")
        print(f">>> Môi trường CheXagent thiếu {pkgs}: cài thêm rồi chạy lại")
        chex_install(py, *pkgs)
    raise RuntimeError("CheXagent vẫn thiếu gói sau 10 lần cài thêm, xem log ngay phía trên")

In [ ]:
done = prep.load_jsonl(REPORTS)
todo = [i for i in lesion_ids if i not in done]
print(f"{len(done)} ảnh đã có báo cáo, còn {len(todo)}")
if todo and hours_left() > 0.5:
    try:
        run_chexagent(todo, time.time() + (hours_left() - 0.3) * 3600)
    except Exception as e:  # không dừng notebook: ô cuối vẫn đóng gói ảnh PNG và báo cáo đã có
        print("LỖI CheXagent:", repr(e), "| các ô sau sẽ bỏ qua, ô cuối vẫn lưu tiến độ. Gửi log này vào thread.")
# Script phụ chỉ ghi báo cáo thô; chuyển thành văn bản ở đây (tính lại mỗi lần, theo SCP_BOXES) rồi ghi lại cả hai trường
rows = prep.load_jsonl(REPORTS)
with open(REPORTS, "w") as f:
    for k, v in rows.items():
        v["report"] = to_report(v["raw"])
        f.write(json.dumps({"image_id": k, "raw": v["raw"], "report": v["report"]}) + "\n")
reports = {k: v["report"] for k, v in rows.items()}
for k in list(reports)[:3]:
    print(k, "|", reports[k][:200])
READY = all(i in reports for i in lesion_ids)
print(len(reports), "báo cáo", "| đủ, làm tiếp" if READY else "| CHƯA ĐỦ: Save Version lần nữa với output phiên này làm input")

### Kiểm tra rò rỉ nhãn
CheXagent-2 trả về tên bệnh theo đúng 14 lớp VinDr kèm hộp toạ độ. Ô dưới chấm riêng các hộp đó như một bộ phát hiện trên tập test, để biết gợi ý chứa bao nhiêu thông tin nhãn.

In [ ]:
# Kiểm tra rò rỉ nhãn: dùng riêng hộp CheXagent làm bộ phát hiện trên tập test 8:1:1 (điểm tin cậy = 1).
# Bài ghi CheXagent "không dùng VinDr-CXR để train"; nếu AP ở đây cao thì gợi ý đã chứa gần như nhãn thật.
if READY:
    name2cls = {n.lower(): i for i, n in enumerate(prep.VINDR_CLASSES)}
    chex_pred, n_unk = {}, 0
    for iid in splits["test"]:
        w, h = meta[iid]
        for name, (x1, y1, x2, y2) in parse_grounding(rows[iid]["raw"]):
            c = name2cls.get(name.lower())
            if c is None:
                m = prep.match_class(name); c = name2cls.get(m.lower()) if m else None
            if c is None:
                n_unk += 1; continue
            chex_pred.setdefault(iid, []).append([c, x1 / 100 * w, y1 / 100 * h, x2 / 100 * w, y2 / 100 * h, 1.0])
    gt_t = cxr_eval.read_boxes(RES_DIR / "gt_test.csv", False)
    thr = [round(x, 2) for x in np.arange(0.40, 0.951, 0.05)]
    ev_c = cxr_eval.Evaluator(gt_t, chex_pred, splits["test"], 14, thr, {"all": (0, float("inf"))})
    tp = fp = fn = 0
    for iid in splits["test"]:
        p, t = {b[0] for b in chex_pred.get(iid, [])}, {b[0] for b in gt_t.get(iid, [])}
        tp += len(p & t); fp += len(p - t); fn += len(t - p)
    print(f"CheXagent một mình trên {len(splits['test'])} ảnh test: AP40 {100 * ev_c.map_over([0.4]):.1f}, "
          f"mAP40:95 {100 * ev_c.map_over(thr):.1f} (bài: SP-Det 42.4 / 21.0)")
    print(f"Lớp bệnh theo ảnh: precision {tp / max(tp + fp, 1):.2f}, recall {tp / max(tp + fn, 1):.2f}; {n_unk} hộp không khớp lớp nào")
    leak = pd.Series({"AP40": 100 * ev_c.map_over([0.4]), "mAP40:95": 100 * ev_c.map_over(thr),
                      "cls_precision": tp / max(tp + fp, 1), "cls_recall": tp / max(tp + fn, 1)}).round(3)
    leak.to_csv(RES_DIR / "chexagent_leak_check.csv")
else:
    print("Bỏ qua ô này: chưa đủ báo cáo CheXagent.")

## 3. Gợi ý tên bệnh (DBP): trích từ báo cáo

Lấy cụm danh từ, loại phần bị phủ định ("no pneumothorax"), rồi khớp với 14 lớp VinDr bằng từ đồng nghĩa. Cụm khớp được đổi thành tên lớp (gợi ý dương); cụm không khớp được giữ làm gợi ý âm, như bài mô tả.
Bảng cuối đo chất lượng gợi ý: tỉ lệ lớp được gợi ý mà ảnh thật sự có. Số này dùng lại cho mục 5.2.

In [ ]:
if READY:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    DBP_FILE = WORK / "dbp.json"
    dbp = json.load(open(DBP_FILE)) if DBP_FILE.exists() else {}
    for iid, rep in reports.items():
        if iid not in dbp:
            dbp[iid] = prep.extract_dbp(rep, nlp)
    json.dump(dbp, open(DBP_FILE, "w"))

    name2id = {n: i for i, n in enumerate(prep.VINDR_CLASSES)}
    tp = fp = fn = 0
    for iid in splits["train"] + splits["val"]:
        pred = {name2id[c] for c in dbp[iid]["positives"]}
        true = {c for c, *_ in fused[iid]}
        tp += len(pred & true); fp += len(pred - true); fn += len(true - pred)
    print(f"Lớp được gợi ý đúng: precision {tp / max(tp + fp, 1):.2f}, recall {tp / max(tp + fn, 1):.2f} (train + val)")
    print("Ví dụ:", json.dumps(dbp[lesion_ids[0]], ensure_ascii=False)[:400])
else:
    print('Bỏ qua ô này: chưa đủ báo cáo CheXagent.')

## 4. Mã hoá gợi ý: báo cáo qua BERT-base (token), tên bệnh qua CLIP ViT-B/32

In [ ]:
if READY:
    prep.encode_prompts(reports, {k: v["prompts"] for k, v in dbp.items()}, PROMPT_DIR,
                        device="cuda" if torch.cuda.is_available() else "cpu")
    print(len(list(PROMPT_DIR.glob("*.pt"))), "file gợi ý")
else:
    print('Bỏ qua ô này: chưa đủ báo cáo CheXagent.')

## 5. Train ba mô hình (cùng cấu hình, cùng seed)

In [ ]:
from ultralytics import YOLO, YOLOWorld
import spdet_model as S
S.CTX.store = S.PromptStore(PROMPT_DIR)
RUNS = WORK / "runs_step1"

def train(kind, seed):
    name = f"{kind}_s{seed}"
    best, done = RUNS / name / "weights" / "best.pt", RUNS / name / "DONE"
    if done.exists():
        print("Đã có", best); return best
    if (RUNS / name).exists():  # lần trước bị ngắt giữa chừng: best.pt chưa phải kết quả cuối, train lại từ đầu
        import shutil; shutil.rmtree(RUNS / name)
    args = dict(data=str(DATA_YAML), seed=seed, device=DEVICE, project=str(RUNS), name=name, exist_ok=True, **TRAIN_ARGS)
    if kind == "yolov8s":
        m = YOLO("yolov8s.pt"); m.train(**args)
    elif kind == "yolo_world":
        m = YOLOWorld("yolov8s-worldv2.pt"); m.train(**args)
    elif kind == "spdet":
        m = YOLOWorld("yolov8s-worldv2.pt"); m.train(trainer=S.SPDetTrainer, **args)
    done.touch()
    return Path(m.trainer.best)

BEST = {}
for seed in SEEDS:
    for kind, on in RUN.items():
        if not on or not READY:
            continue
        if (RUNS / f"{kind}_s{seed}" / "DONE").exists() or hours_left() > EST_HOURS[kind]:
            t = time.time(); BEST[(kind, seed)] = train(kind, seed)
            print(f"{kind} s{seed}: {(time.time() - t) / 3600:.2f} giờ, còn {hours_left():.1f} giờ")
        else:
            print(f"Để {kind} s{seed} cho phiên sau: còn {hours_left():.1f} giờ, cần khoảng {EST_HOURS[kind]} giờ")
BEST

## 6. Đánh giá trên tập test 8:1:1

Mỗi mô hình xuất dự đoán (conf 0.001, tối đa 100 hộp mỗi ảnh) ra CSV theo toạ độ ảnh gốc, rồi chấm bằng `eval/cxr_eval.py`, cùng một bộ hộp cho mọi chỉ số.
Bảng đầu đặt cạnh Bảng 1 của bài. P, R lấy từ `model.val` của Ultralytics, như bài.

In [ ]:
if BEST:
    PAPER = {  # Bảng 1 của bài: P, R, AP40, AP50, AP60, AP70, AP80, AP90, mAP40:95
        "yolov8s":    [45.7, 41.2, 40.3, 35.3, 28.0, 15.4, 6.6, 0.8, 19.4],
        "yolo_world": [46.7, 39.2, 40.1, 35.6, 29.0, 18.3, 7.4, 2.5, 20.5],
        "spdet":      [46.4, 43.8, 42.4, 38.2, 29.5, 18.7, 6.6, 0.7, 21.0],
    }
    COLS = ["P", "R", "AP40", "AP50", "AP60", "AP70", "AP80", "AP90", "mAP40:95"]
    test_imgs = [DS_DIR / "images" / "test" / f"{i}.png" for i in splits["test"]]
    gt = cxr_eval.read_boxes(RES_DIR / "gt_test.csv", False)
    thrs = [round(x, 2) for x in np.arange(0.40, 0.951, 0.05)]
    rows_out, evs = [], {}
    for (kind, seed), best in BEST.items():
        sp = kind == "spdet"
        model = YOLOWorld(str(best)) if kind != "yolov8s" else YOLO(str(best))
        pred_csv = S.export_predictions(model, test_imgs, meta, RES_DIR / f"pred_{kind}_s{seed}.csv", use_prompts=sp,
                                        imgsz=TRAIN_ARGS["imgsz"], device=DEVICE)
        v = model.val(data=str(DATA_YAML), split="test", imgsz=TRAIN_ARGS["imgsz"], batch=16, device=DEVICE,
                      conf=0.001, iou=0.7, plots=False, project=str(RUNS), name=f"val_{kind}_s{seed}", exist_ok=True,
                      **({"validator": S.SPDetValidator} if sp else {}))
        ev = cxr_eval.Evaluator(gt, cxr_eval.read_boxes(pred_csv, True), splits["test"], 14, thrs, {"all": (0, float("inf"))})
        evs[(kind, seed)] = ev
        ours = [100 * v.box.mp, 100 * v.box.mr] + [100 * ev.map_over([t]) for t in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9)] + [100 * ev.map_over(thrs)]
        rows_out.append(pd.Series(ours, COLS, name=f"{kind} s{seed} (nhóm)").round(1))
        rows_out.append(pd.Series(PAPER[kind], COLS, name=f"{kind} (bài)"))
    table1 = pd.DataFrame(rows_out)
    table1.to_csv(RES_DIR / "table1_reproduced.csv")
    display(table1)
else:
    print('Bỏ qua ô này: chưa có mô hình nào train xong.')

In [ ]:
if BEST:
    # So từng lớp với Bảng 2 của bài (AP40)
    PAPER_T2 = {
        "yolov8s":    [44.0, 49.6, 42.5, 73.1, 24.1, 29.4, 33.8, 23.4, 34.5, 13.4, 45.9, 27.1, 73.9, 35.2],
        "yolo_world": [50.0, 47.3, 45.2, 72.5, 23.2, 26.8, 41.0, 22.4, 36.3, 14.6, 51.0, 27.5, 63.5, 31.5],
        "spdet":      [49.6, 48.0, 54.3, 69.8, 33.8, 33.3, 36.1, 25.0, 34.8, 15.6, 42.4, 28.5, 86.5, 32.6],
    }
    t2 = {"Số hộp test": evs[next(iter(evs))].gt_count["all"].sum(0).astype(int)}
    for (kind, seed), ev in evs.items():
        t2[f"{kind} s{seed}"] = (100 * ev.ap_table(0.4)).round(1)
        t2[f"{kind} (bài)"] = PAPER_T2[kind]
    table2 = pd.DataFrame(t2, index=prep.VINDR_CLASSES)
    table2.to_csv(RES_DIR / "table2_reproduced.csv")
    display(table2)
else:
    print('Bỏ qua ô này: chưa có mô hình nào train xong.')

In [ ]:
# Đủ chỉ số Bảng 3 của review + kiểm định cặp SP-Det với từng mô hình nền (bootstrap 1,000 lần theo ảnh)
import subprocess
ids_file = RES_DIR / "gt_test_ids.txt"
for (kind, seed) in BEST:
    if kind == "spdet":
        continue
    a, b = RES_DIR / f"pred_spdet_s{seed}.csv", RES_DIR / f"pred_{kind}_s{seed}.csv"
    if a.exists():
        subprocess.run([sys.executable, str(SPDET_ROOT / "eval" / "cxr_eval.py"), "--gt", str(RES_DIR / "gt_test.csv"),
                        "--pred", str(a), "--pred-b", str(b), "--images", str(ids_file),
                        "--out", str(RES_DIR / f"spdet_vs_{kind}_s{seed}")], check=True)
        print(open(RES_DIR / f"spdet_vs_{kind}_s{seed}.md").read().split("## Kiem dinh cap")[-1])

## 8. Đóng gói tiến độ cho phiên sau
Ô này luôn phải chạy tới, kể cả khi các ô trên bỏ qua.

In [ ]:
# Đóng gói tiến độ để phiên sau dùng lại, và chép bảng kết quả ra ngoài cho dễ tải
out_res = Path("/kaggle/working/results_step1")
shutil.copytree(RES_DIR, out_res, dirs_exist_ok=True)
t = time.time(); tmp = SAVE_TAR.with_suffix(".part")
with tarfile.open(tmp, "w") as tf:
    tf.add(WORK, arcname=WORK.name)
tmp.rename(SAVE_TAR)
print(f"Đã ghi {SAVE_TAR} ({SAVE_TAR.stat().st_size / 1e9:.1f} GB) trong {time.time() - t:.0f}s")
todo_models = [f"{k} s{s}" for s in SEEDS for k, on in RUN.items() if on and not (WORK / "runs_step1" / f"{k}_s{s}" / "DONE").exists()]
print("Còn phải train:", ", ".join(todo_models) if todo_models else "không, bước 1 đã xong. Gửi results_step1/table1_reproduced.csv và table2_reproduced.csv vào thread.")

## 7. Kết luận bước 1

- Nếu mAP40:95 của SP-Det cách 21.0 không quá khoảng 1 điểm và thứ tự SP-Det > YOLO-World > YOLOv8s giữ nguyên: coi như tái lập được, chuyển sang bước 2 (tập test chính thức).
- Nếu lệch nhiều: ghi rõ mức lệch trong bài (review yêu cầu), liệt kê các lựa chọn ở đầu notebook như nguồn chênh lệch có thể có, và cân nhắc liên hệ tác giả để hỏi imgsz, cách gộp hộp, cách chia.
- Tập test 8:1:1 chỉ khoảng 440 ảnh, không có ảnh bình thường, nên AUC mức ảnh chưa tính được. Bước 2 sẽ sửa điều này.

Gửi `results_step1/table1_reproduced.csv` và `table2_reproduced.csv` vào thread để Claude tick mục 1.1 đến 1.3.